In [6]:
import requests, time
import pandas as pd

BASE_V1 = "https://api.elections.kalshi.com/v1"

In [7]:
def get_all_atp_events():
    events, page = [], 1
    while True:
        r = requests.get(f"{BASE_V1}/events",
                         params={"series_ticker": "KXATPMATCH", "limit": 100, "page_number": page})
        batch = r.json().get("events", [])
        if not batch:
            break
        events.extend(batch)
        dates = sorted(e.get("target_datetime", "")[:10] for e in batch)
        print(f"page {page}: {len(batch)} events ({dates[0]} to {dates[-1]})")
        page += 1
        time.sleep(0.1)
    return events

all_atp_events = get_all_atp_events()
print(f"\nTotal: {len(all_atp_events)} events")

page 1: 100 events (2026-05-24 to 2026-05-30)
page 2: 100 events (2026-05-18 to 2026-05-24)
page 3: 100 events (2026-05-16 to 2026-05-18)
page 4: 100 events (2026-05-06 to 2026-05-16)
page 5: 100 events (2026-04-24 to 2026-05-07)
page 6: 100 events (2026-04-15 to 2026-04-24)
page 7: 100 events (2026-04-08 to 2026-04-15)
page 8: 100 events (2026-04-01 to 2026-04-08)
page 9: 100 events (2026-03-22 to 2026-04-01)
page 10: 100 events (2026-03-16 to 2026-03-22)
page 11: 100 events (2026-03-04 to 2026-03-16)
page 12: 100 events (2026-02-23 to 2026-03-04)
page 13: 100 events (2026-02-18 to 2026-02-23)
page 14: 100 events (2026-02-14 to 2026-02-18)
page 15: 100 events (2026-02-08 to 2026-02-14)
page 16: 100 events (2026-01-23 to 2026-02-08)
page 17: 100 events (2026-01-18 to 2026-01-23)
page 18: 100 events (2026-01-13 to 2026-01-18)
page 19: 100 events (2026-01-10 to 2026-01-13)
page 20: 100 events (2025-12-18 to 2026-01-10)
page 21: 100 events (2025-10-30 to 2025-12-18)
page 22: 100 events (2

In [9]:
competitions = {}
for e in all_atp_events:
    comp = e.get("product_metadata", {}).get("competition", "N/A")
    competitions[comp] = competitions.get(comp, 0) + 1

print(f"{'Competition':<45} {'Events':>6}")
print("-" * 53)
for comp, count in sorted(competitions.items()):
    print(f"{comp:<45} {count:>6}")

Competition                                   Events
-----------------------------------------------------
ATP Acapulco                                      45
ATP Adelaide                                      40
ATP Almaty                                        28
ATP Athens                                        36
ATP Auckland                                      43
ATP Barcelona                                     51
ATP Basel                                         39
ATP Bastad                                        27
ATP Beijing                                       31
ATP Brisbane                                      56
ATP Brussels                                      27
ATP Bucharest                                     40
ATP Buenos Aires                                  39
ATP Chengdu                                       28
ATP Cincinnati                                    95
ATP Dallas                                        45
ATP Delray Beach                             

In [5]:
rows = []
for e in all_atp_events:
    event_fields = {f"event_{k}": v for k, v in e.items() if k != "markets"}
    for m in e.get("markets", []):
        row = {**event_fields, **{f"market_{k}": v for k, v in m.items()}}
        rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("kalshi_atp_match_markets.csv", index=False)
print(f"Saved {len(df)} rows ({df['event_ticker'].nunique()} matches), {len(df.columns)} columns")
df.head()

Saved 6529 rows (3265 matches), 77 columns


,event_id,event_ticker,event_series_ticker,event_target_datetime,event_mutually_exclusive,event_mutually_exclusive_side,event_collateral_return_type,event_title,event_category,event_tags,...,market_expected_expiration_date,market_strike_type,market_custom_strike,market_component_leg_title,market_price_level_structure,market_liquidity,market_previous_day_price,market_previous_day_price_dollars,market_previous_week_price,market_previous_week_price_dollars
0,7cf03938-38de-4fbf-9125-aa6f1fcf2acc,KXATPMATCH-26MAY30KOUTAB,KXATPMATCH,2026-05-30T12:00:00Z,True,yes,MECNET,Kouame vs Tabilo,Sports,None,...,2026-05-30T12:00:00Z,structured,{'tennis_competitor': 'a82a9f1f-838b-4c40-850f...,Moise Kouame,linear_cent,0,NaN,NaN,NaN,NaN
1,7cf03938-38de-4fbf-9125-aa6f1fcf2acc,KXATPMATCH-26MAY30KOUTAB,KXATPMATCH,2026-05-30T12:00:00Z,True,yes,MECNET,Kouame vs Tabilo,Sports,None,...,2026-05-30T12:00:00Z,structured,{'tennis_competitor': '3e22c35d-63bc-44ff-9ee5...,Alejandro Tabilo,linear_cent,0,NaN,NaN,NaN,NaN
2,4e8f16bf-c14a-4f83-b4f7-f8f06c3c8b0b,KXATPMATCH-26MAY30FARTIA,KXATPMATCH,2026-05-30T12:00:00Z,True,yes,MECNET,Faria vs Tiafoe,Sports,None,...,2026-05-30T12:00:00Z,structured,{'tennis_competitor': '667e04df-9b7e-46af-9e6f...,Jaime Faria,linear_cent,0,NaN,NaN,NaN,NaN
3,4e8f16bf-c14a-4f83-b4f7-f8f06c3c8b0b,KXATPMATCH-26MAY30FARTIA,KXATPMATCH,2026-05-30T12:00:00Z,True,yes,MECNET,Faria vs Tiafoe,Sports,None,...,2026-05-30T12:00:00Z,structured,{'tennis_competitor': '35b9acda-2a7c-4255-aaf0...,Frances Tiafoe,linear_cent,0,NaN,NaN,NaN,NaN
4,d0d40110-ee22-4c26-97c6-671c2bb1d2f1,KXATPMATCH-26MAY30COBTIE,KXATPMATCH,2026-05-30T12:00:00Z,True,yes,MECNET,Cobolli vs Tien,Sports,None,...,2026-05-30T12:00:00Z,structured,{'tennis_competitor': '05040645-e988-4e97-a807...,Flavio Cobolli,linear_cent,0,NaN,NaN,NaN,NaN


In [12]:

# Pull KXAOMEN series
def get_all_events(series_ticker):
    events, page = [], 1
    while True:
        r = requests.get(f"{BASE_V1}/events",
                         params={"series_ticker": series_ticker, "limit": 100, "page_number": page})
        batch = r.json().get("events", [])
        if not batch:
            break
        events.extend(batch)
        dates = sorted(e.get("target_datetime", "")[:10] for e in batch)
        print(f"page {page}: {len(batch)} events ({dates[0]} to {dates[-1]})")
        page += 1
        time.sleep(0.1)
    return events

aomen_events = get_all_events("KXAOMEN")
print(f"\nTotal KXAOMEN events: {len(aomen_events)}")


page 1: 1 events (2026-01-12 to 2026-01-12)

Total KXAOMEN events: 1


In [13]:

# Show KXAOMEN matches and cross-reference with KXATPMATCH
aomen_df = pd.DataFrame([
    {"event_ticker": e["ticker"],
     "event_title": e.get("title", ""),
     "target_date": e.get("target_datetime", "")[:10],
     "round": e.get("product_metadata", {}).get("round", "N/A"),
     "num_markets": len(e.get("markets", []))}
    for e in aomen_events
]).sort_values("target_date")

print(f"KXAOMEN total events: {len(aomen_df)}")
print(f"\nBy round:")
print(aomen_df["round"].value_counts().to_string())

# Which KXAOMEN tickers are NOT in KXATPMATCH (i.e. missing from our original pull)
atpmatch_tickers = set(pd.read_csv("kalshi_atp_match_markets.csv")["event_ticker"].unique())
aomen_only = aomen_df[~aomen_df["event_ticker"].isin(atpmatch_tickers)]
print(f"\nEvents in KXAOMEN but NOT in KXATPMATCH ({len(aomen_only)}):")
print(aomen_only[["event_ticker", "event_title", "target_date", "round"]].to_string(index=False))


KXAOMEN total events: 1

By round:
round
N/A    1

Events in KXAOMEN but NOT in KXATPMATCH (1):
event_ticker                                event_title target_date round
  KXAOMEN-26 Australian Open Final: Alcaraz vs Djokovic  2026-01-12   N/A


In [14]:

# Find all Kalshi series that contain ATP match-level events (not tournament winner/outright markets)
import re

series_df = pd.read_csv("kalshi_sports_series.csv")

# Filter to plausible tennis series by keyword
tennis_keywords = r"ATP|WTA|tennis|open men|grand slam|australian open|french open|wimbledon|us open"
candidates = series_df[series_df["title"].str.contains(tennis_keywords, case=False, na=False)
                      | series_df["ticker"].str.contains(tennis_keywords, case=False, na=False)]
print(f"Candidate series: {len(candidates)}")
print(candidates.to_string(index=False))


Candidate series: 88
                  ticker                                               title
                 KXAOMEN                                Australian Open Mens
          KXAOMENSINGLES                        Argentina Open men's singles
               KXAOWOMEN                              Australian Open Womens
                   KXATP                             Men's Tournament Winner
              KXATP1RANK                                ATP #1 Ranked Player
                KXATPAMT ATP Tour Abierto Mexican Telcel presentado por HSBC
             KXATPANYSET                                  ATP Any Set Winner
    KXATPCHALLENGERMATCH                                     Challenger ATP 
            KXATPDOUBLES                            ATP Doubles Tennis Match
         KXATPEXACTMATCH                               ATP Exact Match Score
          KXATPEXACTSETS                                             testing
             KXATPFINALS                               

In [15]:

# Probe each candidate series: pull 1 event and check if it's a head-to-head match
# A match event has competition_scope == "Game" and a "vs" title
match_series = []

for _, row in candidates.iterrows():
    ticker = row["ticker"]
    r = requests.get(f"{BASE_V1}/events", params={"series_ticker": ticker, "limit": 1})
    events = r.json().get("events", [])
    if not events:
        continue
    e = events[0]
    title = e.get("title", "") or e.get("event_title", "")
    meta = e.get("product_metadata", {}) or {}
    scope = meta.get("competition_scope", "")
    league = meta.get("league", "")
    is_match = scope == "Game" or " vs " in title or " v " in title
    if is_match:
        match_series.append({"ticker": ticker, "title": row["title"], "sample_event": title, "scope": scope, "league": league})
    time.sleep(0.05)

match_series_df = pd.DataFrame(match_series)
print(f"Series with match-level events: {len(match_series_df)}")
print(match_series_df.to_string(index=False))


Series with match-level events: 25
              ticker                            title                                           sample_event             scope                       league
             KXAOMEN             Australian Open Mens             Australian Open Final: Alcaraz vs Djokovic              Game                          ATP
           KXAOWOMEN           Australian Open Womens           Australian Open Final: Sabalenka vs Rybakina            Future                          WTA
               KXATP          Men's Tournament Winner                       ATP Hamburg Finals: Buse vs Paul              Game                          ATP
KXATPCHALLENGERMATCH                  Challenger ATP                                           Smith vs Zink              Game               ATP Challenger
        KXATPDOUBLES         ATP Doubles Tennis Match                 Pavlasek / Rikl vs Nouza / Oberleitner              Game                          ATP
     KXATPEXACTMATCH         

In [ ]:

# Cell 1: Create kalshi_candles.db with markets + candles schema
import sqlite3, requests, pandas as pd, ast, time
from datetime import datetime, timezone

DB_PATH = "kalshi/data/kalshi_candles.db"
BASE_V2 = "https://external-api.kalshi.com/trade-api/v2"

con = sqlite3.connect(DB_PATH)
con.executescript("""
CREATE TABLE IF NOT EXISTS markets (
    market_ticker TEXT PRIMARY KEY, market_id TEXT,
    event_ticker TEXT, event_id TEXT, series_ticker TEXT,
    event_title TEXT, player TEXT,
    competition TEXT, round TEXT, league TEXT,
    scheduled_start TEXT, occurrence_datetime TEXT,
    create_date TEXT, list_date TEXT, open_date TEXT,
    close_date TEXT, expiration_date TEXT,
    status TEXT, result TEXT, settlement_value REAL,
    yes_bid REAL, yes_ask REAL, last_price REAL,
    prev_day_price REAL, prev_week_price REAL,
    volume REAL, open_interest REAL, dollar_volume REAL, liquidity REAL,
    downloaded_at TEXT
);
CREATE TABLE IF NOT EXISTS candles (
    market_ticker   TEXT NOT NULL,
    end_period_ts   TEXT NOT NULL,  -- ISO timestamp string
    open            REAL,           -- price.open_dollars (NULL if no trading that minute)
    high            REAL,           -- price.high_dollars
    low             REAL,           -- price.low_dollars
    close           REAL,           -- price.close_dollars
    mean            REAL,           -- price.mean_dollars
    previous        REAL,           -- last known close (reconstructed via ffill)
    volume          REAL,           -- volume_fp
    open_interest   REAL,           -- open_interest_fp
    bid_close       REAL,           -- yes_bid.close_dollars
    ask_close       REAL,           -- yes_ask.close_dollars
    PRIMARY KEY (market_ticker, end_period_ts),
    FOREIGN KEY (market_ticker) REFERENCES markets(market_ticker)
);
CREATE INDEX IF NOT EXISTS idx_candles_market ON candles(market_ticker);
CREATE INDEX IF NOT EXISTS idx_candles_ts ON candles(end_period_ts);
""")
con.commit()
print("DB ready:", DB_PATH)


In [ ]:

# Reset: drop and recreate candles table to apply updated schema
# (Run this once before Cell 3 if candles table already exists from a prior run)
con.execute("DROP TABLE IF EXISTS candles")
con.executescript("""
CREATE TABLE candles (
    market_ticker   TEXT NOT NULL,
    end_period_ts   TEXT NOT NULL,
    open            REAL,
    high            REAL,
    low             REAL,
    close           REAL,
    mean            REAL,
    previous        REAL,
    volume          REAL,
    open_interest   REAL,
    bid_close       REAL,
    ask_close       REAL,
    PRIMARY KEY (market_ticker, end_period_ts),
    FOREIGN KEY (market_ticker) REFERENCES markets(market_ticker)
);
CREATE INDEX IF NOT EXISTS idx_candles_market ON candles(market_ticker);
CREATE INDEX IF NOT EXISTS idx_candles_ts ON candles(end_period_ts);
""")
con.commit()
print("candles table recreated with updated schema")


In [17]:

# Cell 2: Import all market metadata from CSV into markets table
df = pd.read_csv("kalshi_atp_match_markets.csv")

def parse_meta(s):
    try:
        d = ast.literal_eval(s) if isinstance(s, str) else {}
        return d.get("competition"), d.get("round"), d.get("league")
    except:
        return None, None, None

meta = df["event_product_metadata"].apply(parse_meta)
df["competition"] = [m[0] for m in meta]
df["round"]       = [m[1] for m in meta]
df["league"]      = [m[2] for m in meta]

keep = df[[
    "market_ticker_name", "market_id", "event_ticker", "event_id", "event_series_ticker",
    "event_title", "market_component_leg_title",
    "competition", "round", "league",
    "event_target_datetime", "market_event_occurrence_datetime",
    "market_create_date", "market_list_date", "market_open_date",
    "market_close_date", "market_expiration_date",
    "market_status", "market_result", "market_expiration_value",
    "market_yes_bid", "market_yes_ask", "market_last_price",
    "market_previous_day_price", "market_previous_week_price",
    "market_volume", "market_open_interest", "market_dollar_volume", "market_liquidity",
]].rename(columns={
    "market_ticker_name": "market_ticker",
    "event_series_ticker": "series_ticker",
    "market_component_leg_title": "player",
    "event_target_datetime": "scheduled_start",
    "market_event_occurrence_datetime": "occurrence_datetime",
    "market_create_date": "create_date",
    "market_list_date": "list_date",
    "market_open_date": "open_date",
    "market_close_date": "close_date",
    "market_expiration_date": "expiration_date",
    "market_status": "status",
    "market_result": "result",
    "market_expiration_value": "settlement_value",
    "market_yes_bid": "yes_bid",
    "market_yes_ask": "yes_ask",
    "market_last_price": "last_price",
    "market_previous_day_price": "prev_day_price",
    "market_previous_week_price": "prev_week_price",
    "market_volume": "volume",
    "market_open_interest": "open_interest",
    "market_dollar_volume": "dollar_volume",
    "market_liquidity": "liquidity",
})
keep["downloaded_at"] = None

keep.to_sql("markets", con, if_exists="replace", index=False)
con.commit()
print(f"Inserted {len(keep)} markets")
print(keep[["market_ticker","event_title","competition","round","player"]].head())


Inserted 6529 markets
                  market_ticker       event_title              competition  \
0  KXATPMATCH-26MAY30KOUTAB-KOU  Kouame vs Tabilo  French Open Men Singles   
1  KXATPMATCH-26MAY30KOUTAB-TAB  Kouame vs Tabilo  French Open Men Singles   
2  KXATPMATCH-26MAY30FARTIA-FAR   Faria vs Tiafoe  French Open Men Singles   
3  KXATPMATCH-26MAY30FARTIA-TIA   Faria vs Tiafoe  French Open Men Singles   
4  KXATPMATCH-26MAY30COBTIE-COB   Cobolli vs Tien  French Open Men Singles   

         round            player  
0  Round Of 32      Moise Kouame  
1  Round Of 32  Alejandro Tabilo  
2  Round Of 32       Jaime Faria  
3  Round Of 32    Frances Tiafoe  
4  Round Of 32    Flavio Cobolli  


In [ ]:

# Cell 3: Download candles for all markets into DB
# Window: scheduled_start - 2 days → scheduled_start + 8 hours (≤3360 min, API max=5000)
# Response flattening: price{open/high/low/close/mean/previous}_dollars, yes_bid, yes_ask

def parse_candles(raw, market_ticker):
    rows = []
    for c in raw:
        p = c.get("price", {})
        bid = c.get("yes_bid", {})
        ask = c.get("yes_ask", {})
        rows.append({
            "market_ticker": market_ticker,
            "end_period_ts":  pd.Timestamp(c["end_period_ts"], unit="s", tz="UTC").isoformat(),
            "open":           float(p["open_dollars"])     if "open_dollars"  in p else None,
            "high":           float(p["high_dollars"])     if "high_dollars"  in p else None,
            "low":            float(p["low_dollars"])      if "low_dollars"   in p else None,
            "close":          float(p["close_dollars"])    if "close_dollars" in p else None,
            "mean":           float(p["mean_dollars"])     if "mean_dollars"  in p else None,
            "previous":       float(p["previous_dollars"]) if "previous_dollars" in p else None,
            "volume":         float(c.get("volume_fp", 0) or 0),
            "open_interest":  float(c.get("open_interest_fp", 0) or 0),
            "bid_close":      float(bid["close_dollars"])  if "close_dollars" in bid else None,
            "ask_close":      float(ask["close_dollars"])  if "close_dollars" in ask else None,
        })
    cdf = pd.DataFrame(rows)
    # Reconstruct 'previous' for every row: carry last close forward
    # (mirrors old CSV behavior where previous was always populated)
    cdf["previous"] = cdf["previous"].fillna(cdf["close"].ffill())
    return cdf

def fetch_candles(series, ticker, scheduled_start):
    match_ts = pd.Timestamp(scheduled_start)
    start_ts = int((match_ts - pd.Timedelta(days=2)).timestamp())
    end_ts   = int((match_ts + pd.Timedelta(hours=8)).timestamp())
    params   = {"start_ts": start_ts, "end_ts": end_ts, "period_interval": 1}
    for url in [
        f"{BASE_V2}/series/{series}/markets/{ticker}/candlesticks",
        f"{BASE_V2}/historical/markets/{ticker}/candlesticks",
    ]:
        try:
            r = requests.get(url, params=params, timeout=15)
            if r.ok:
                return r.json().get("candlesticks", [])
            # Log non-OK but don't retry on 4xx (except 400 which might be range issue)
        except Exception:
            pass
    return None

CANDLE_COLS = ["market_ticker","end_period_ts","open","high","low","close",
               "mean","previous","volume","open_interest","bid_close","ask_close"]

errors = []
rows = keep.to_dict("records")

for i, row in enumerate(rows):
    ticker  = row["market_ticker"]
    series  = row["series_ticker"]
    sched   = row["scheduled_start"]

    raw = fetch_candles(series, ticker, sched)

    if raw is None:
        errors.append({"ticker": ticker, "error": "both endpoints failed"})
    elif raw:
        cdf = parse_candles(raw, ticker)
        cdf[CANDLE_COLS].to_sql("candles", con, if_exists="append", index=False, method="ignore")

    con.execute("UPDATE markets SET downloaded_at=? WHERE market_ticker=?",
                (datetime.now(timezone.utc).isoformat(), ticker))

    if i % 200 == 0:
        con.commit()
        print(f"{i}/{len(rows)}  errors: {len(errors)}")
    time.sleep(0.05)

con.commit()
print(f"\nDone. {len(errors)} failed tickers.")
for e in errors[:20]:
    print(e)


In [ ]:

# Cell 4: Verify the DB
print("markets:   ", con.execute("SELECT COUNT(*) FROM markets").fetchone()[0])
print("candles:   ", con.execute("SELECT COUNT(*) FROM candles").fetchone()[0])
print("downloaded:", con.execute("SELECT COUNT(*) FROM markets WHERE downloaded_at IS NOT NULL").fetchone()[0])
print("no data:   ", con.execute("SELECT COUNT(*) FROM markets WHERE downloaded_at IS NOT NULL AND market_ticker NOT IN (SELECT DISTINCT market_ticker FROM candles)").fetchone()[0])
print()
print("Rounds coverage:")
print(pd.read_sql("SELECT competition, round, COUNT(*) as n FROM markets GROUP BY competition, round ORDER BY competition, n DESC", con).to_string(index=False))
print()
pd.read_sql("SELECT * FROM candles LIMIT 5", con)
